# Week 3：Back-propagation and Learning（讲解版）

本笔记根据 `wk03/03-backprop-1.pdf` 的所有知识点整理，逐条解释概念，并以可渲染 LaTeX 的 Markdown 编写。

## 1. 概览（Overview）

本节课的主题包含：
- 监督学习与经验风险最小化复习
- 多层感知机（MLP）
- 端到端计算图
- 自动微分（AD），正向/反向模式
- 反向传播、前向与反向两次传递
- 梯度消失与爆炸
- 内存布局与梯度存储
- 学习流程整合
- 参数初始化
- 随机梯度下降（SGD）
- 迭代、epoch 与数据打乱
- 模型评估与选择
- SGD 加速方法：动量、AdaGrad、Adam、AdamW
- 学习率与学习率调度


## 2. 监督学习回顾（Supervised Learning Review）

- 参数化模型：
$$ y = f(x; \theta) $$
- 训练集：
$$ \mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^N $$
- 正则化损失函数：
$$ L(\theta) = \sum_{i=1}^N \ell(f(x^{(i)};\theta), y^{(i)}) + R(\theta) $$
其中 $R(\theta)$ 是先验/正则项。

- 优化目标：
$$ \theta^* = \arg\min_{\theta} L(\theta) $$

- 深度学习中 $L$ 通常 **非凸**，所以常接受局部最优。

- 最简单的优化方法：梯度下降
$$ \theta \leftarrow \theta - \eta \nabla_{\theta} L $$
其中 $\eta$ 是学习率。


## 3. 多层感知机（MLP）

- 最基本的深度学习模型：多层感知机。
- 输入、隐藏层、输出：
  - 输入 $x \in \mathbb{R}^{p_0}$
  - 隐藏层 $z_j \in \mathbb{R}^{p_j}$
  - 输出 $y \in \mathbb{R}^{p_n}$

- 每一层计算：
$$ z_j = f_j(z_{j-1}) = \sigma_j(A_j z_{j-1} + b_j) $$
  - $A_j \in \mathbb{R}^{p_j \times p_{j-1}}$
  - $b_j \in \mathbb{R}^{p_j}$
  - $\sigma_j$ 是逐元素激活函数

- 网络整体是函数复合：
$$ y = (f_n \circ \cdots \circ f_2 \circ f_1)(x) $$


## 4. 端到端计算图（End-to-end Computation Graph）

深度学习把模型看成由很多简单“结点/算子”组成的计算图：
- 每个结点是一个参数化函数。
- 整体是从输入到输出的复合映射。

计算图视角是理解**前向计算**与**反向传播**的核心工具。


## 5. 函数求值（Forward Evaluation）

前向计算就是按拓扑顺序依次算出中间变量：

$$ z_1 = f_1(x; \theta_1) $$
$$ z_2 = f_2(z_1; \theta_2) $$
$$ \cdots $$
$$ y = f_8(z_4, z_7; \theta_8) $$

这些中间量在反向传播中通常要被用到，因此会被缓存。


## 6. 梯度求值（Gradient Evaluation）

反向传播基于链式法则：

- **单路径例子**：
$$ \frac{\partial L}{\partial \theta_7} = \frac{\partial L}{\partial y}\frac{\partial y}{\partial z_7}\frac{\partial z_7}{\partial \theta_7} $$

- **多路径例子**（有分叉时要“求和”）：
$$ \frac{\partial L}{\partial \theta_1} = \frac{\partial L}{\partial y}\frac{\partial y}{\partial z_4}\frac{\partial z_4}{\partial z_3}\frac{\partial z_3}{\partial z_2}\frac{\partial z_2}{\partial z_1}\frac{\partial z_1}{\partial \theta_1} + \frac{\partial L}{\partial y}\frac{\partial y}{\partial z_7}\frac{\partial z_7}{\partial z_6}\frac{\partial z_6}{\partial z_5}\frac{\partial z_5}{\partial z_1}\frac{\partial z_1}{\partial \theta_1} $$

核心思想：**一条路径就是一串乘积，多条路径就求和。**


## 7. 多元微分记号（Notational Aside）

- 标量函数 $f: \mathbb{R}^n \to \mathbb{R}$ 的梯度：
$$ \nabla f(x) = \left[\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}\right] \in \mathbb{R}^n $$

- 向量函数 $f: \mathbb{R}^n \to \mathbb{R}^m$ 的雅可比矩阵：
$$ J_f(x) = \frac{\partial f}{\partial x} \in \mathbb{R}^{m\times n} $$

- 文献中经常把 $d$ 和 $\partial$ 混用（“标准太多”），需要理解具体上下文。


## 8. 自动微分（Automatic Differentiation, AD）

- 自动微分是**算法化的求导**：给出精确导数的代码。
- 假设计算由一组“基础算子”组成：
  - 加减乘除
  - $\exp, \log$
  - 三角函数等

- AD 有两种模式：
  - **正向模式（Forward mode）**：固定一个输入变量 $u$，求所有输出 $v$ 的导数 $\frac{dv}{du}$。
  - **反向模式（Reverse mode）**：固定一个输出 $v$，求所有输入 $u$ 的导数 $\frac{dv}{du}$。

深度学习里通常损失是标量、参数很多，因此更偏好反向模式。


## 9. 正向 vs 反向 AD 的效率

- **正向模式**更适合“输入少、输出多”的情况。
- **反向模式**更适合“输出少（常是标量损失）、输入多（参数很多）”的情况。

反向模式的核心操作是 **向量-雅可比积（Vector-Jacobian Product, VJP）**，可以避免显式构造大雅可比矩阵。


## 10. 正向模式的双数（Dual Numbers）

正向模式可以用“双数”实现：

- 把变量替换为 $v + \Delta v$，其中 $(\Delta v)^2 = 0$。
- 设置 $\Delta u = 1$，则：
$$ \Delta v = \frac{dv}{du} $$

**例 1：** $y=ax+b$

$$ y + \Delta y = a(x+\Delta x)+b \Rightarrow \Delta y = a\Delta x $$

**例 2：** $y=\exp(x)$

用泰勒展开可得到：
$$ \Delta y = \Delta x \exp(x) $$

这说明双数传播能“随前向计算一起”得到导数。


## 11. 梯度计算的代价与顺序

在深度学习里，我们往往需要对 **所有参数** 同时更新。
- 正向模式要为每个参数都走一遍，代价高。
- 反向模式一次反向就能得到所有参数的梯度。

因此，深度学习通常采用 **反向模式 AD（即 backprop）**。


## 12. 反向传播（Back-propagation）

- 在深度学习里，**反向模式 AD 就是反向传播**。
- 不同框架实现略有差别：
  - 显式构建计算图
  - 或通过算子追踪隐式构图

概念上，对每行前向代码：
```
P, Q = foo(A, B, C)
```
自动微分会生成反向代码：
```
dLdA, dLdB, dLdC = foo_vjp(A, B, C, P, Q, dLdP, dLdQ)
```

反向传播需要两次传递：
- **Forward pass**：算输出并缓存中间量
- **Backward pass**：计算梯度


## 13. 深度学习结点（Layer）的前向/反向

一个结点（层）包含：
- 输入 $x$
- 参数 $\theta$
- 输出 $y=f(x,\theta)$

**前向**：
$$ y = f(x, \theta) $$

**反向**：给定 $\frac{\partial L}{\partial y}$，计算：
$$ \frac{\partial L}{\partial x},\; \frac{\partial L}{\partial \theta} $$

伪代码：
```
y = f(x, theta)
L = loss(y, y_true)
L.backward()  # 计算所有梯度
```


## 14. 结点示例：$y=\sqrt{1/x}$

- 前向：
$$ y = \sqrt{\frac{1}{x}} = x^{-1/2} $$

- 反向：
$$ \frac{dy}{dx} = -\frac{1}{2}x^{-3/2} = -\frac{1}{2}y^3 $$

因此：
$$ \frac{dL}{dx} = \frac{dL}{dy} \cdot \frac{dy}{dx} = -\frac{1}{2}y^3 \frac{dL}{dy} $$


## 15. 批数据与 VJP（Vector-Jacobian Products）

假设：
- 输入维度 $n$
- 输出维度 $m$
- 参数维度 $p$

则：
- $\frac{\partial y}{\partial x}$ 是 $m\times n$
- $\frac{\partial y}{\partial \theta}$ 是 $m\times p$

反向传播时常用 **向量-雅可比积**：
- $\frac{dL}{dy}$ 是 $1\times m$
- 最终得到 $\frac{dL}{dx}$（$1\times n$）和 $\frac{dL}{d\theta}$（$1\times p$）

**关键点：** 实际实现中不显式构造雅可比矩阵，而直接算 VJP。

对批数据（batch size = $N$）：
$$ \frac{dL}{d\theta} = \sum_{i=1}^N \frac{dL}{dy^{(i)}}\frac{dy^{(i)}}{d\theta} $$


## 16. 梯度消失与爆炸

- 深层网络中梯度是连乘：
$$ \frac{dL}{d\theta} = \frac{dL}{dy} \frac{dy}{dz} \cdots \frac{dz}{d\theta} $$

- 如果连乘中因子 < 1，梯度会快速衰减（**消失**）。
- 如果因子 > 1，梯度会快速增大（**爆炸**）。

**现象更易发生在 sigmoid 类激活函数。**

研究界开发了许多缓解方法（例如：更好的初始化、归一化、残差结构等）。


## 17. 内存开销（Concerning Memory）

- **参数**占内存相对少。
- **梯度**大小与数据同量级（常以转置形式存储）。
- 前向时 **in-place 操作** 可节省内存。
- 反向时 **复用缓冲区** 可节省内存。
- 测试阶段只有前向，不必存中间结果。
- 数据常以批形式处理：例如 $B\times C \times \cdots \times N$。


## 18. PyTorch 的自定义 Autograd Function

在 PyTorch 中可以手写 forward/backward：

```python
class InvSqrtFcn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        with torch.no_grad():
            y = 1.0 / torch.sqrt(x)
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, dLdy):
        if dLdy is None:
            return None
        (y,) = ctx.saved_tensors
        dLdx = -0.5 * torch.pow(y, 3.0) * dLdy
        return dLdx
```

关键点：
- `ctx.save_for_backward` 存储前向量以供反向使用。
- backward 实现的是 **VJP**。


## 19. Clone 与 Detach

- `clone()`：复制一份张量并**保留计算图**，梯度可回传。
- `detach()`：共享数据但**切断梯度**。

```python
y_hat = y.clone()   # 计算图被复制

y_hat = y.detach()  # 共享内存，但不参与梯度
```

这对控制梯度流向很重要。


## 20. Putting It All Together（学习流程）

完整训练流程：
1. 构建模型（网络结构 + 参数初始化）。
2. 前向计算得到预测。
3. 计算损失函数。
4. 反向传播得到梯度。
5. 用优化算法更新参数。
6. 重复直到收敛或达到停止条件。


## 21. Iris 分类示例

- Iris 数据集（Fisher, 1936）
- 3 类：Setosa, Versicolour, Virginica
- 4 个特征：萼片长度/宽度、花瓣长度/宽度
- 150 个样本（每类 50）

模型：
- 输入 4 维
- 隐藏层 8 个节点（ReLU）
- 输出 3 维（softmax）

结构：
$$ x \xrightarrow{A,b} z \xrightarrow{\text{ReLU}} z' \xrightarrow{C,d} \hat{y} \xrightarrow{\text{softmax}} y $$

训练：
- 交叉熵损失
$$ \ell(\theta) = -\log[\text{softmax}(f(x;\theta))]_y $$
- 梯度下降更新：
$$ \theta \leftarrow \theta - \eta \nabla L(\theta) $$


## 22. PyTorch 中的模型定义

两种写法：

**Sequential：**
```python
model = nn.Sequential(
    nn.Linear(4, 8, bias=True),
    nn.ReLU(True),
    nn.Linear(8, 3, bias=True)
)
```

**模块类：**
```python
class IrisMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8, bias=True)
        self.layer2 = nn.ReLU(True)
        self.layer3 = nn.Linear(8, 3, bias=True)

    def forward(self, x):
        z1 = self.layer1(x)
        z2 = self.layer2(z1)
        y_hat = self.layer3(z2)
        return y_hat
```

注意：`nn.CrossEntropyLoss` 内部包含 softmax，因此模型最后一层可以直接输出 logits。对于预测，取 `argmax(logits)` 与 `argmax(softmax(logits))` 等价。


## 23. 参数初始化

- 通常用随机分布：$\mathcal{N}(0,\sigma^2)$ 或 $U(-\sigma, \sigma)$
- 偏置常设为 0
- **Xavier / Glorot 初始化**：
$$ \sigma = \sqrt{\frac{2}{n_i + n_o}} $$
- **Kaiming 初始化**：
$$ \sigma = \sqrt{\frac{2}{n_i}} \quad \text{或} \quad \sigma = \sqrt{\frac{2}{n_o}} $$

目标：保持特征方差在网络中大致不变。

PyTorch 在创建模型时会自动初始化参数。


## 24. 学习曲线（Learning Curves）

学习曲线通常绘制：
- 训练损失 vs 迭代数
- 分类错误率 vs 迭代数

可以看到模型逐步优于随机基线。


## 25. 随机梯度下降（SGD）

完整梯度：
$$ \nabla L(\theta) = \frac{1}{N} \sum_{i=1}^N \nabla_\theta \ell(f(x^{(i)};\theta), y^{(i)}) $$

SGD 用 mini-batch $I$ 近似：
$$ \nabla \widehat{L}(\theta) = \frac{1}{|I|} \sum_{i\in I} \nabla_\theta \ell(f(x^{(i)};\theta), y^{(i)}) $$

要点：
- $|I|$ = batch size
- 在温和假设下，$\mathbb{E}[\nabla \widehat{L}] = \nabla L$
- 可跨多个 batch 累积梯度
- 通常会打乱数据后按顺序分 batch

术语：
- 每次参数更新叫一次 **iteration**
- 整个数据集过一遍叫一个 **epoch**


## 26. 学习曲线：SGD vs GD

- GD（全量梯度）每次更新成本大，但方向更稳定。
- SGD 噪声更大，但更快、更可扩展。

图上通常展示：
- batch size = 10
- 10 iterations = 1 epoch


## 27. 随机种子（Random Seeds）

- 深度网络是非凸的。
- 不同随机初始化、不同 batch 顺序会导致不同结果。

因此常用随机种子控制可复现性，并报告多次运行的均值与标准差。


## 28. 模型选择（Choosing the Best Model）

- 最后一轮参数 **不一定最好**。
- 标准做法：划分数据集
  - **训练集**：更新参数
  - **验证集**：选模型/调超参数
  - **测试集**：最终评估泛化能力（最好只用一次）


## 29. 改善泛化性能

常见方法：
- 正则化（经典方法）
- 早停（缺少验证集时也可用）
- 数据增强（加噪声/扰动）
- 收集更多真实数据
- 大规模预训练 + 任务微调


## 30. 数据白化（Data Whitening）与归一化

- 白化：
$$ x = \Sigma^{-1/2}(x-\mu) $$
常能改善收敛与泛化。

- 大规模数据中常做 **逐维归一化**：
$$ x_i = \frac{x_i - \mu_i}{\sigma_i} $$

- 在深度网络中常通过 **BatchNorm** 实现逐层归一化，并维护测试时的统计量。


## 31. 学习率（Learning Rate）

- 学习率 $\eta$ 是关键超参数。
- 过小：收敛慢；过大：可能发散。
- 学习曲线展示不同学习率的差异。


## 32. 学习率调度（Learning Rate Schedules）

常见调度：
- **常数**：$\eta(t)=\eta_0$
- **线性**：$\eta(t)=\eta_{\text{init}} + \frac{t}{t_{\max}}(\eta_{\text{final}}-\eta_{\text{init}})$
- **阶梯**：
$$ \eta(t) = \begin{cases}
\gamma \eta(t-1), & t \% t_{\text{step}} = 0 \\
\eta(t-1), & \text{otherwise}
\end{cases} $$
- **余弦**：
$$ \eta(t) = \eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi t / t_{\text{period}})) $$

也可组合，例如 **线性 warm-up + 余弦衰减**。

注意：学习率阶跃变化常导致训练损失突然上升。


## 33. 动量与 SGD 变体

动量（Momentum）：
$$ g(t) = \nabla L(\theta(t)) + \mu g(t-1) $$
$$ \theta(t+1) = \theta(t) - \eta g(t) $$

动量可加速收敛并减少震荡。

其他常见变体：AdaGrad、Adam、AdamW。


## 34. AdaGrad

思想：为不同特征自适应步长。

视作一阶 proximal 方法：
$$ \theta(t+1) = \arg\min_\theta \langle \nabla L(\theta(t)), \theta \rangle + \frac{1}{2\eta}\|\theta-\theta(t)\|^2 $$

等价更新：
$$ \theta(t+1) = \theta(t) - \eta B_t^{-1}\nabla L(\theta(t)) $$

其中 $B_t$ 用历史梯度估计（对角矩阵）：
$$ G(t) = \left(\sum_{k=1}^t \text{diag}(\nabla L(\theta(k))^2) + \epsilon I\right)^{1/2} $$

**缺点**：$G(t)$ 不断增大，学习率会趋于极小。


## 35. Adam

Adam 维护一阶与二阶动量：
$$ m(t) = \beta_1 m(t-1) + (1-\beta_1)\nabla L(\theta(t)) $$
$$ v(t) = \beta_2 v(t-1) + (1-\beta_2)(\nabla L(\theta(t)))^2 $$

偏差修正：
$$ \hat{m}(t) = \frac{m(t)}{1-\beta_1^t}, \quad \hat{v}(t) = \frac{v(t)}{1-\beta_2^t} $$

更新：
$$ \theta(t+1) = \theta(t) - \eta \, \text{diag}(\hat{v}(t)+\epsilon)^{-1/2} \hat{m}(t) $$


## 36. AdamW

AdamW = Adam + **权重衰减（weight decay）**：
$$ \theta(t+1) = \theta(t) - \eta \, \text{diag}(\hat{v}(t)+\epsilon)^{-1/2} \hat{m}(t) - \eta w\,\theta(t) $$

- 与对 $m(t)$ 做 $\ell_2$ 正则化不同。
- 实证效果更好，因此目前常作为默认优化器。


## 37. 优化器学习曲线对比

常见比较：
- SGD
- SGD + Momentum
- AdamW
- SGD + 学习率调度

通常 AdamW 或带调度的 SGD 收敛更快、更稳定。
